# s16 — Session Resume

**What this teaches:** how to reuse a session across multiple turns so the model retains memory of earlier turns. ADK already persists the event log of an active session in memory; you just need to pass **the same user ID and session ID** to consecutive `Run` calls.

**Concept anchor:** a *session* is the unit of conversational memory. `RunOnce(ctx, r, prompt)` quietly uses an auto-generated session; for explicit control you call `r.Run(ctx, userID, sessionID, content, cfg)` and choose the IDs yourself. Two turns under the same `(userID, sessionID)` share the same history.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- Any configured LLM provider. No tools, plugins, or extra deps required.
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```

## 1. Imports

We reach directly into `genai` (the content type) and `adk/agent` (`RunConfig`) because we are calling the lower-level `r.Run` instead of `agentkit.RunOnce`.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "google.golang.org/genai"

    "google.golang.org/adk/agent"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/stream"
)

## 2. Helper

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the model, agent, and runner

Nothing different from [s01_loop](../s01_loop/s01_loop.ipynb) yet — the magic happens in step 4.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s16_resume",
    Description: "Session-resume demo (in-memory, two turns).",
    Model:       llm,
})
must(err)
r, err := agentkit.Runner("s16", a)
must(err)

## 4. Define a `turn` helper

Each call to `turn` issues a user message into the **same** `(userID="u", sessionID="sess")` slot. Because the runner's in-memory session store is keyed on that pair, every call appends to the same conversation.

In [ ]:
turn := func(text string) {
    seq := r.Run(ctx, "u", "sess",
        &genai.Content{Role: "user", Parts: []*genai.Part{{Text: text}}},
        agent.RunConfig{})
    _ = stream.Print(os.Stdout, seq)
}

## 5. Two turns, one session

First we teach the model a fact, then we ask it back. If the session is being reused correctly, the answer to turn 2 should reference the fact from turn 1 — without us re-sending it.

In [ ]:
turn("Remember: my favourite colour is teal.")
fmt.Println("--- new turn (same session) ---")
turn("What is my favourite colour?")

## What to look for

- The second turn answers "teal" without you re-supplying the fact — that is the resume working. Change `"sess"` to a different string in one of the two calls and the second turn will plead amnesia.
- Persistence here is *in-memory*. For across-process resume you need a backing store (the HTTP server in `server/` does this via the `conversation_<id>.json` files described in `CLAUDE.md`). Pair with [s18_events](../s18_events/s18_events.ipynb) to see `session_start`/`session_end` events.

## Try it yourself

1. Add a third `turn(...)` call asking an unrelated follow-up — confirm the model still has the teal fact several turns in.
2. Change one of the calls to `userID="v"` (different user, same session ID) and observe that the runner treats it as a fresh conversation.